# 01_train.ipynb — Heston DDN 模型训练

本 Notebook 负责：
1. 加载（或生成）200k 训练数据
2. 按 70:15:15 划分数据集
3. 训练 HestonDDN（论文超参数严格对齐）
4. 评估测试集误差
5. 保存模型权重至 `models/heston_ddn_weights.pth`

In [1]:
## Section 1: 导入依赖与项目路径配置
import sys
import os
from pathlib import Path

# 将 modules 目录加入 Python 搜索路径
PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
# 当在 Notebook 中运行时，PROJECT_ROOT 为 Heston_Project/
MODULES_DIR = PROJECT_ROOT / "modules"
sys.path.insert(0, str(PROJECT_ROOT))

from modules.model import HestonDDN
from modules.pricing import generate_training_data

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ---- 路径常量 ----
DATA_PATH  = PROJECT_ROOT / "data" / "heston_dataset_200k.csv"
MODELS_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODELS_DIR / "heston_ddn_weights.pth"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_PATH    : {DATA_PATH}")
print(f"MODEL_PATH   : {MODEL_PATH}")

PROJECT_ROOT : /Users/liaojiansong/calibration/Zhang_realize
DATA_PATH    : /Users/liaojiansong/calibration/Zhang_realize/data/heston_dataset_200k.csv
MODEL_PATH   : /Users/liaojiansong/calibration/Zhang_realize/models/heston_ddn_weights.pth


In [2]:
## Section 2: 加载或生成训练数据集

REQUIRED_COLS = ['kappa','lambda','sigma','rho','v0','r','tau','S0',
                 'K','Price','log_K_S0','price_norm',
                 'd_kappa','d_lambda','d_sigma','d_rho','d_v0']

# 若本地 data/ 目录下没有 CSV，则尝试从上级目录寻找
FALLBACK_PATH = Path("/Users/liaojiansong/calibration/heston_dataset_200k.csv")

if DATA_PATH.exists():
    src = DATA_PATH
elif FALLBACK_PATH.exists():
    src = FALLBACK_PATH
    print(f"⚠️  data/ 下无 CSV，使用备用路径: {src}")
else:
    src = None

if src is not None:
    print(f"📂 正在加载: {src}")
    df_all = pd.read_csv(src)
    # 验证必需列
    missing = [c for c in REQUIRED_COLS if c not in df_all.columns]
    if missing:
        raise ValueError(f"CSV 缺少以下列，请重新生成数据: {missing}")
    print(f"✅ 加载完成: {len(df_all):,} 条，列: {list(df_all.columns)}")
else:
    print("⚠️  未找到 CSV，正在生成 200k 训练数据（需较长时间）...")
    df_all = generate_training_data(num_samples=200_000)
    df_all.to_csv(DATA_PATH, index=False)
    print(f"💾 数据已保存至: {DATA_PATH}")

# 关键列统计
print()
print(df_all[['price_norm','log_K_S0','d_kappa','d_sigma']].describe().round(4))

📂 正在加载: /Users/liaojiansong/calibration/Zhang_realize/data/heston_dataset_200k.csv
✅ 加载完成: 199,989 条，列: ['kappa', 'lambda', 'sigma', 'rho', 'v0', 'r', 'tau', 'S0', 'K', 'Price', 'd_kappa', 'd_lambda', 'd_sigma', 'd_rho', 'd_v0', 'log_K_S0', 'price_norm']

        price_norm     log_K_S0      d_kappa      d_sigma
count  199989.0000  199989.0000  199989.0000  199989.0000
mean        0.2151      -0.0000       0.0012      -0.0068
std         0.1337       0.2887       0.0126       0.0107
min         0.0000      -0.5000      -0.0825      -0.1104
25%         0.1008      -0.2500      -0.0033      -0.0112
50%         0.2089      -0.0000       0.0001      -0.0038
75%         0.3269       0.2500       0.0043       0.0004
max         0.5562       0.5000       0.3130       0.0289


In [3]:
## Section 3: 数据集划分（70:15:15）与 Tensor 转换

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
BATCH_SIZE  = 256

device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"使用设备: {device}")

# 随机打乱
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)
n_total = len(df_all)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)

df_train = df_all.iloc[:n_train]
df_val   = df_all.iloc[n_train : n_train + n_val]
df_test  = df_all.iloc[n_train + n_val :]

print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")

# 特征与梯度列定义
feat_cols = ['kappa','lambda','sigma','rho','v0','r','tau','S0','log_K_S0']
grad_cols = ['d_kappa','d_lambda','d_sigma','d_rho','d_v0']

def df_to_tensors(df, dev):
    X = torch.tensor(df[feat_cols].values, dtype=torch.float32).to(dev)
    y = torch.tensor(df['price_norm'].values, dtype=torch.float32).view(-1,1).to(dev)
    g = torch.tensor(df[grad_cols].values,  dtype=torch.float32).to(dev)
    return X, y, g

X_train, y_train, g_train = df_to_tensors(df_train, device)
X_val,   y_val,   g_val   = df_to_tensors(df_val,   device)
X_test,  y_test,  g_test  = df_to_tensors(df_test,  device)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

使用设备: mps
Train: 139,992  Val: 29,998  Test: 29,999
X_train shape: torch.Size([139992, 9]), y_train shape: torch.Size([139992, 1])


In [4]:
## Section 4: 模型初始化与归一化 Scaler 拟合

EPOCHS        = 200
LEARNING_RATE = 0.001
DECAY_GAMMA   = 0.9
DECAY_STEP    = 20   # 每 20 epoch 学习率 × 0.9

model = HestonDDN(input_dim=9).to(device)
model.fit_scalers(X_train, y_train)   # 冻结归一化极值（register_buffer）

train_dataset = TensorDataset(X_train, y_train, g_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=DECAY_STEP, gamma=DECAY_GAMMA)

print(f"模型参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"批次数/epoch: {len(train_loader)}")
print(f"归一化极值 y_min={model.y_min.item():.4f}, y_max={model.y_max.item():.4f}")

模型参数量: 114,901
批次数/epoch: 547
归一化极值 y_min=0.0000, y_max=0.5562


In [5]:
## Section 5: 训练循环（200 Epochs）

t0 = time.time()
print(f"{'Epoch':>6s} | {'Train Loss':>12s} | {'Price loss':>10s} | {'grad loss':>10s} | {'Val RelErr%':>10s} | {'LR':>10s} | {'Elapsed':>8s}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    l_p = 0.0
    l_g = 0.0
    for b_X, b_y, b_g in train_loader:
        optimizer.zero_grad()
        loss, l_p, l_g = model.compute_loss(b_X, b_y, b_g)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        l_p += l_p.item()
        l_g += l_g.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    l_p_avg = l_p / len(train_loader)
    l_g_avg = l_g / len(train_loader)

    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            pred_val_norm = model(X_val)
            S0_val = X_val[:, 7:8]
            pred_val_abs = pred_val_norm * S0_val
            y_val_abs    = y_val * S0_val
            rel_err = (torch.abs(pred_val_abs - y_val_abs) /
                       (torch.abs(y_val_abs) + 1e-3) * 100)
            mean_rel = rel_err.mean().item()
        elapsed = time.time() - t0
        lr_now  = scheduler.get_last_lr()[0]
        print(f"{epoch:6d} | {avg_loss:12.6f} | {l_p_avg:10.6f} | {l_g_avg:10.6f} | {mean_rel:12.4f} | {lr_now:10.6f} | {elapsed:7.1f}s")

print(f"\n⏱ 训练完成，总耗时: {time.time()-t0:.1f}s")

 Epoch |   Train Loss | Price loss |  grad loss | Val RelErr% |         LR |  Elapsed
------------------------------------------------------------
     1 |     0.039386 |   0.000012 |   0.000025 |   33801.0547 |   0.001000 |     3.2s
    20 |     0.000442 |   0.000000 |   0.000001 |    4501.0659 |   0.000900 |    59.0s
    40 |     0.000173 |   0.000000 |   0.000000 |    5090.3076 |   0.000810 |   120.3s
    60 |     0.000092 |   0.000000 |   0.000000 |     580.5141 |   0.000729 |   180.0s
    80 |     0.000033 |   0.000000 |   0.000000 |    1775.3829 |   0.000656 |   240.0s
   100 |     0.000196 |   0.000000 |   0.000000 |     202.4198 |   0.000590 |   296.2s
   120 |     0.000037 |   0.000000 |   0.000000 |     877.6995 |   0.000531 |   352.2s
   140 |     0.000013 |   0.000000 |   0.000000 |     303.7999 |   0.000478 |   410.6s
   160 |     0.000012 |   0.000000 |   0.000000 |     911.1266 |   0.000430 |   468.6s
   180 |     0.000012 |   0.000000 |   0.000000 |    1040.2885 |   0.0

In [6]:
## Section 6: 测试集评估与误差统计

model.eval()
with torch.no_grad():
    pred_test_norm = model(X_test)
    S0_test        = X_test[:, 7:8]
    pred_test_abs  = pred_test_norm * S0_test
    y_test_abs     = y_test * S0_test

    abs_err  = torch.abs(pred_test_abs - y_test_abs)
    rel_err  = abs_err / (torch.abs(y_test_abs) + 1e-3) * 100

# 前 15 条样本明细
print(f"  {'#':>5s} | {'真实价格':>13s} | {'预测价格':>13s} | {'绝对误差':>10s} | {'相对误差':>8s}")
print("  " + "-"*60)
for i in range(min(15, len(y_test_abs))):
    tp = y_test_abs[i].item()
    pp = pred_test_abs[i].item()
    ae = abs_err[i].item()
    re = rel_err[i].item()
    flag = "✅" if re < 5.0 else "❌"
    print(f"  {flag}{i+1:4d} | {tp:13.4f} | {pp:13.4f} | {ae:10.4f} | {re:7.2f}%")

# 全量统计
print()
print(f"  📊 Test set ({len(y_test_abs):,} 条) 统计:")
print(f"     均值相对误差   : {rel_err.mean().item():.4f}%")
print(f"     中位数相对误差 : {rel_err.median().item():.4f}%")
print(f"     最大相对误差   : {rel_err.max().item():.4f}%")
print(f"     相对误差 < 1%  : {(rel_err < 1.0).float().mean().item()*100:.1f}%")
print(f"     相对误差 < 5%  : {(rel_err < 5.0).float().mean().item()*100:.1f}%")

      # |          真实价格 |          预测价格 |       绝对误差 |     相对误差
  ------------------------------------------------------------
  ✅   1 |        4.6912 |        4.7544 |     0.0632 |    1.35%
  ✅   2 |     1856.7809 |     1865.0253 |     8.2444 |    0.44%
  ✅   3 |      696.8992 |      710.3440 |    13.4448 |    1.93%
  ❌   4 |       28.4322 |       37.8604 |     9.4283 |   33.16%
  ✅   5 |     1718.8654 |     1726.8074 |     7.9420 |    0.46%
  ✅   6 |      929.8089 |      934.1285 |     4.3196 |    0.46%
  ✅   7 |     1367.8667 |     1374.8374 |     6.9707 |    0.51%
  ❌   8 |       17.3786 |       18.7255 |     1.3469 |    7.75%
  ✅   9 |      405.3498 |      423.2195 |    17.8697 |    4.41%
  ✅  10 |     1531.6461 |     1537.8893 |     6.2432 |    0.41%
  ✅  11 |      151.8329 |      155.9547 |     4.1218 |    2.71%
  ✅  12 |     1606.8224 |     1614.1632 |     7.3408 |    0.46%
  ✅  13 |      107.0555 |      110.5221 |     3.4666 |    3.24%
  ❌  14 |      192.6381 |      202.7315 |

In [7]:
## Section 7: 保存模型权重

torch.save({
    'model_state_dict': model.state_dict(),   # 包含 register_buffer 的归一化极值
    'is_fitted':  model.is_fitted,
    'input_dim':  model.input_dim,
    'heston_dim': model.heston_dim,
}, MODEL_PATH)

size_kb = MODEL_PATH.stat().st_size / 1024
print(f"💾 模型已保存: {MODEL_PATH}  ({size_kb:.1f} KB)")
print(f"   归一化极值:")
print(f"     X_min = {model.X_min.cpu().numpy()}")
print(f"     X_max = {model.X_max.cpu().numpy()}")
print(f"     y_min = {model.y_min.item():.6f}")
print(f"     y_max = {model.y_max.item():.6f}")
print()
print("✅ 训练全流程完成！")

💾 模型已保存: /Users/liaojiansong/calibration/Zhang_realize/models/heston_ddn_weights.pth  (455.2 KB)
   归一化极值:
     X_min = [ 1.0011792e-02  1.0004548e-02  1.0000127e-01 -8.9999771e-01
  1.0001554e-02  1.0003702e-03  5.0008450e-02  1.0028770e+01
 -4.9999633e-01]
     X_max = [ 4.9999971e+00  9.9999273e-01  9.9998772e-01 -5.0001502e-02
  9.9998927e-01  9.9999733e-02  9.9999821e-01  5.9999736e+03
  4.9999893e-01]
     y_min = 0.000000
     y_max = 0.556161

✅ 训练全流程完成！
